Exercise

MultHeadAttentiom Implementation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0, "Embedding dim must be divisible by num_heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # Combined projection for Q, K, V
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        # 1. Linear Projection: (B, S, D) -> (B, S, 3*D)
        qkv = self.qkv_proj(x)

        # 2. Reshape & Transpose for Multi-head: (B, S, 3, H, head_D) -> (3, B, H, S, head_D)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2] # Each is (B, H, S, head_D)

        # 3. Scaled Dot-Product Attention
        # (B, H, S, head_D) @ (B, H, head_D, S) -> (B, H, S, S)
        attn_logits = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            # Mask is usually (1, 1, S, S) or (B, 1, S, S)
            attn_logits = attn_logits.masked_fill(mask == 0, float("-inf"))

        attn_weights = F.softmax(attn_logits, dim=-1)

        # 4. Context Vector: (B, H, S, S) @ (B, H, S, head_D) -> (B, H, S, head_D)
        context = attn_weights @ v

        # 5. Concatenate Heads: (B, H, S, head_D) -> (B, S, D)
        context = context.transpose(1, 2).reshape(batch_size, seq_len, self.embed_dim)

        return self.out_proj(context)

# --- Test Run ---
B, S, D, H = 2, 10, 64, 8
mha = MultiHeadAttention(D, H)
x = torch.randn(B, S, D)
output = mha(x)
print(f"Output shape: {output.shape}") # Should be (2, 10, 64)


Output shape: torch.Size([2, 10, 64])


Transformer implementation

In [ ]:
import torch
import torch.nn as nn
import math

class CustomTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_layers, ff_dim, max_len=100):
        super().__init__()

        # 1. Standard PyTorch Embedding
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        # 2. Standard PyTorch Position Embedding
        # Using nn.Embedding to learn positions (common in models like BERT/GPT)
        self.pos_embedding = nn.Embedding(max_len, d_model)

        # 3. Transformer Layers using nn.MultiheadAttention
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ff_dim)
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        batch_size, seq_len = x.size()

        # Create position indices: [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(x.device)

        # Combine Token and Position Embeddings
        x = self.token_embedding(x) + self.pos_embedding(positions)
        x = self.dropout(x)

        # Pass through Transformer blocks
        for layer in self.layers:
            x = layer(x)

        return self.fc_out(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim):
        super().__init__()

        # Official PyTorch MultiheadAttention
        self.attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )

    def forward(self, x):
        # Attention + Residual
        # nn.MultiheadAttention returns (output, weights)
        attn_out, _ = self.attention(query=x, key=x, value=x)
        x = self.norm1(x + attn_out)

        # Feed Forward + Residual
        ff_out = self.feed_forward(x)
        x = self.norm2(x + ff_out)
        return x

# --- Execution ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomTransformer(vocab_size=1000, d_model=256, n_heads=8, num_layers=4, ff_dim=512).to(device)

# Dummy Input [batch_size, seq_len]
src = torch.randint(0, 1000, (32, 20)).to(device)
output = model(src)

print(f"Device: {device}")
print(f"Output Shape: {output.shape}") # [32, 20, 1000]


Device: cpu
Output Shape: torch.Size([32, 20, 1000])


In [ ]:
Diffusion DDPM implementation:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. THE DIT MODEL (The "Brain") ---
class TimeSeriesDiT(nn.Module):
    def __init__(self, seq_len=100, feature_dim=12, hidden_size=128):
        super().__init__()
        self.input_projection = nn.Linear(feature_dim, hidden_size)
        self.time_embed = nn.Sequential(nn.Linear(1, hidden_size), nn.SiLU(), nn.Linear(hidden_size, hidden_size))
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len + 1, hidden_size))

        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        self.output_projection = nn.Linear(hidden_size, feature_dim)

    def forward(self, x, t):
        # t is a scalar tensor of time steps
        t_emb = self.time_embed(t.float().unsqueeze(-1)).unsqueeze(1)
        x = self.input_projection(x)
        x = torch.cat((t_emb, x), dim=1) + self.pos_embed
        x = self.transformer(x)
        return self.output_projection(x[:, 1:, :])

# --- 2. THE DIFFUSION ENGINE ---
class MarketDiffusion:
    def __init__(self, steps=100, device="cuda"):
        self.steps = steps
        self.device = device
        # Define a noise schedule (Linear)
        self.beta = torch.linspace(1e-4, 0.02, steps).to(device)
        self.alpha = 1.0 - self.beta
        self.alpha_cumprod = torch.cumprod(self.alpha, dim=0)

    # -------------------------------------------------------
    # NOISING PROCESS (Forward Pass: Data -> Noise)
    # -------------------------------------------------------
    def add_noise(self, x_start, t):
        """Adds noise to clean market data based on timestep t"""
        noise = torch.randn_like(x_start)
        sqrt_alpha_cumprod = torch.sqrt(self.alpha_cumprod[t])[:, None, None]
        sqrt_one_minus_alpha_cumprod = torch.sqrt(1. - self.alpha_cumprod[t])[:, None, None]

        # Formula: x_t = sqrt(alpha_bar)*x_0 + sqrt(1-alpha_bar)*noise
        return sqrt_alpha_cumprod * x_start + sqrt_one_minus_alpha_cumprod * noise, noise

    # -------------------------------------------------------
    # GENERATIVE PROCESS (Backward Pass: Noise -> Data)
    # -------------------------------------------------------
    @torch.no_grad()
    def generate_samples(self, model, shape):
        """Starts with pure noise and uses DiT to remove it step-by-step"""
        model.eval()
        # 1. Start with pure Gaussian noise
        img = torch.randn(shape).to(self.device)

        # 2. Iterate backwards from T to 1
        for i in reversed(range(self.steps)):
            t = torch.full((shape[0],), i, device=self.device, dtype=torch.long)

            # 3. Model predicts the noise present in the current sample
            predicted_noise = model(img, t)

            # 4. Remove a fraction of that noise (Simplified DDPM step)
            alpha = self.alpha[i]
            alpha_cumprod = self.alpha_cumprod[i]
            beta = self.beta[i]

            if i > 0:
                noise = torch.randn_like(img)
            else:
                noise = 0

            img = (1 / torch.sqrt(alpha)) * (img - ((1 - alpha) / (torch.sqrt(1 - alpha_cumprod))) * predicted_noise) + torch.sqrt(beta) * noise

        return img

# --- Colab Execution ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TimeSeriesDiT().to(device)
diffuser = MarketDiffusion(device=device)

# 1. Simulate Noising (Forward)
clean_trades = torch.randn(8, 100, 12).to(device) # Real data
t = torch.randint(0, 100, (8,)).to(device)
noisy_trades, _ = diffuser.add_noise(clean_trades, t)

# 2. Simulate Generation (Backward)
synthetic_trades = diffuser.generate_samples(model, (8, 100, 12))

print(f"Generated synthetic market sequence: {synthetic_trades.shape}")



Generated synthetic market sequence: torch.Size([8, 100, 12])


Diffusion DDIM implementation:

In [ ]:
import torch
import torch.nn as nn
import math

# --- 1. THE DIT MODEL (Same architecture as before) ---
class TimeSeriesDiT(nn.Module):
    def __init__(self, seq_len=100, feature_dim=12, hidden_size=128):
        super().__init__()
        self.input_projection = nn.Linear(feature_dim, hidden_size)
        self.time_embed = nn.Sequential(nn.Linear(1, hidden_size), nn.SiLU(), nn.Linear(hidden_size, hidden_size))
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len + 1, hidden_size))

        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=4, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        self.output_projection = nn.Linear(hidden_size, feature_dim)

    def forward(self, x, t):
        t_emb = self.time_embed(t.float().unsqueeze(-1)).unsqueeze(1)
        x = self.input_projection(x)
        x = torch.cat((t_emb, x), dim=1) + self.pos_embed
        x = self.transformer(x)
        return self.output_projection(x[:, 1:, :])

# --- 2. THE DDIM SAMPLER ---
class DDIMSampler:
    def __init__(self, model, n_steps=1000, device="cuda"):
        self.model = model
        self.device = device
        self.n_steps = n_steps

        # Define the beta schedule
        self.beta = torch.linspace(1e-4, 0.02, n_steps).to(device)
        self.alpha = 1.0 - self.beta
        self.alpha_cumprod = torch.cumprod(self.alpha, dim=0)

    @torch.no_grad()
    def sample(self, shape, ddim_steps=50, eta=0.0):
        """
        ddim_steps: Number of steps to actually run (e.g., 50 instead of 1000)
        eta: 0.0 for deterministic DDIM, 1.0 for stochastic DDPM-like
        """
        self.model.eval()
        batch_size = shape[0]

        # 1. Start with pure noise
        x = torch.randn(shape, device=self.device)

        # 2. Create the subset of timesteps to visit (e.g., [999, 979, 959...])
        times = torch.linspace(0, self.n_steps - 1, ddim_steps).long().to(self.device)
        times = list(reversed(times.cpu().numpy()))
        time_pairs = list(zip(times[:-1], times[1:])) # [(now, next), (next, after_next)...]

        for i, j in time_pairs:
            now_t = torch.full((batch_size,), i, device=self.device, dtype=torch.long)
            next_t = torch.full((batch_size,), j, device=self.device, dtype=torch.long)

            # Get alpha_cumprod for current and next step
            a_t = self.alpha_cumprod[i]
            a_next = self.alpha_cumprod[j]

            # 3. Model predicts noise
            e_t = self.model(x, now_t)

            # 4. Predict x_0 (The denoised signal)
            pred_x0 = (x - torch.sqrt(1 - a_t) * e_t) / torch.sqrt(a_t)

            # 5. Compute the "direction pointing to x_t"
            # In DDIM, we can set sigma to 0 for deterministic sampling
            sigma_t = eta * torch.sqrt((1 - a_next) / (1 - a_t)) * torch.sqrt(1 - a_t / a_next)
            dir_xt = torch.sqrt(1 - a_next - sigma_t**2) * e_t

            # 6. Update x
            x = torch.sqrt(a_next) * pred_x0 + dir_xt

            if eta > 0:
                x += sigma_t * torch.randn_like(x)

        return x

# --- 3. COLAB RUN ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TimeSeriesDiT().to(device)
sampler = DDIMSampler(model, device=device)

# Generate 4 market sequences (100 steps, 12 features) in only 20 steps!
synthetic_data = sampler.sample(shape=(4, 100, 12), ddim_steps=20)

print(f"Device: {device}")
print(f"Final Data Shape: {synthetic_data.shape}")


/tmp/ipykernel_15601/1176383757.py:14: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)


Device: cpu
Final Data Shape: torch.Size([4, 100, 12])


Difussion Flow matching implementation:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. THE FLOW-MATCHING MODEL (DiT) ---
class FlowDiT(nn.Module):
    def __init__(self, seq_len=100, feature_dim=12, hidden_size=128):
        super().__init__()
        self.input_projection = nn.Linear(feature_dim, hidden_size)
        # Time 't' is now 0.0 to 1.0
        self.time_embed = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size))
        self.pos_embed = nn.Parameter(torch.zeros(1, seq_len + 1, hidden_size))

        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_size, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        self.output_projection = nn.Linear(hidden_size, feature_dim)

    def forward(self, x, t):
        # t: (Batch,) normalized 0 to 1
        t_emb = self.time_embed(t.float().unsqueeze(-1)).unsqueeze(1)
        x = self.input_projection(x)
        x = torch.cat((t_emb, x), dim=1) + self.pos_embed
        x = self.transformer(x)
        return self.output_projection(x[:, 1:, :])

# --- 2. THE FLOW-MATCHING ENGINE ---
class FlowMatcher:
    def __init__(self, device="cuda"):
        self.device = device

    # FORWARD PASS: Create the "Straight Line" (Conditional Flow)
    def get_train_tuple(self, x_1):
        """
        x_1: Real market data (Target)
        """
        batch_size = x_1.shape[0]
        x_0 = torch.randn_like(x_1) # Pure noise (Source)

        # Random time step between 0 and 1
        t = torch.rand(batch_size, device=self.device)
        t_reshaped = t[:, None, None]

        # THE FLOW EQUATION (Interpolation)
        # x_t = (1 - t) * x_0 + t * x_1
        x_t = (1.0 - t_reshaped) * x_0 + t_reshaped * x_1

        # THE TARGET VELOCITY (The vector the model must learn)
        # ut = x_1 - x_0
        ut = x_1 - x_0

        return x_t, t, ut

    # GENERATIVE PASS: Solve the ODE (Inference)
    @torch.no_grad()
    def sample(self, model, shape, steps=50):
        model.eval()
        # 1. Start at x_0 (Noise)
        x = torch.randn(shape, device=self.device)
        dt = 1.0 / steps

        # 2. Step forward in time (Euler Integration)
        for i in range(steps):
            t = torch.full((shape[0],), i / steps, device=self.device)

            # Predict the velocity (direction to data)
            v = model(x, t)

            # x_next = x + velocity * dt
            x = x + v * dt

        return x

# --- 3. COLAB RUN ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = FlowDiT().to(device)
matcher = FlowMatcher(device=device)

# TRAIN STEP EXAMPLE
real_data = torch.randn(16, 100, 12).to(device)
x_t, t, target_v = matcher.get_train_tuple(real_data)
pred_v = model(x_t, t)
loss = F.mse_loss(pred_v, target_v) # Model learns the "velocity"

# INFERENCE EXAMPLE
synthetic_trades = matcher.sample(model, (8, 100, 12), steps=20)

print(f"Loss: {loss.item():.4f}")
print(f"Synthetic Output: {synthetic_trades.shape}")


Loss: 2.3244
Synthetic Output: torch.Size([8, 100, 12])


5 Layers model implementation in Pytorch?

In [ ]:
import torch
import torch.nn as nn
class TradeModel5Layer(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=512, output_dim=1, dropout_prob=0.2):
        super().__init__()

        # Layer 1: Input -> Hidden
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_prob)
        )

        # Layer 2: Hidden -> Hidden
        self.layer2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_prob)
        )

        # Layer 3: Hidden -> Hidden
        self.layer3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU()
        )

        # Layer 4: Hidden -> Hidden
        self.layer4 = nn.Sequential(
            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.BatchNorm1d(hidden_dim // 4),
            nn.ReLU()
        )

        # Layer 5: Output Layer
        # Often used to predict Price Change or Volatility
        self.output_layer = nn.Linear(hidden_dim // 4, output_dim)

    def forward(self, x):
        # Pass through the 5-layer stack
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.output_layer(x)


5 Layers Model implementation in Python with dropout?

In [ ]:
import numpy as np

class TradeModel5LayerDropout:
    def __init__(self, input_dim=20, hidden_dim=512, output_dim=1, dropout_p=0.2):
        self.dropout_p = dropout_p
        self.keep_prob = 1 - dropout_p

        # Dimensions: 512 -> 512 -> 256 -> 128 -> output
        self.dims = [input_dim, hidden_dim, hidden_dim, hidden_dim // 2, hidden_dim // 4, output_dim]

        # He Initialization
        self.params = {}
        for i in range(1, 6):
            self.params[f'W{i}'] = np.random.randn(self.dims[i-1], self.dims[i]) * np.sqrt(2./self.dims[i-1])
            self.params[f'b{i}'] = np.zeros((1, self.dims[i]))

    def relu(self, z):
        return np.maximum(0, z)

    def forward(self, X, training=True):
        self.cache = {'a0': X}

        # Layers 1 through 4 (Hidden Layers with ReLU + Dropout)
        for i in range(1, 5):
            # Linear + ReLU
            z = self.cache[f'a{i-1}'] @ self.params[f'W{i}'] + self.params[f'b{i}']
            a = self.relu(z)

            # Inverted Dropout (Only in early layers 1 & 2 for stability)
            if training and i <= 2:
                mask = (np.random.rand(*a.shape) < self.keep_prob) / self.keep_prob
                a *= mask
                self.cache[f'mask{i}'] = mask

            self.cache[f'z{i}'] = z
            self.cache[f'a{i}'] = a

        # Layer 5 (Output Layer - Linear)
        self.out = np.dot(self.cache['a4'], self.params['W5']) + self.params['b5']
        return self.out


In [ ]:
Flash Attention

In [ ]:
import torch

def flash_attention_forward(Q, K, V, block_size_i=64, block_size_j=64):
    """
    Pure Python/PyTorch implementation of the FlashAttention Forward Pass.

    Shapes:
        Q: [N, D] (Sequence Length, Head Dimension)
        K: [N, D]
        V: [N, D]
    """
    N, D = Q.shape
    scale = 1.0 / (D ** 0.5)

    # 1. Initialize the output matrix O in HBM (Global Memory)
    O = torch.zeros_like(Q, device=Q.device)

    # 2. Initialize running softmax statistics for each row
    # m stores the running maximum value (initialized to -infinity)
    m = torch.full((N, 1), float('-inf'), device=Q.device)
    # l stores the running sum of exponential denominators (initialized to 0)
    l = torch.zeros((N, 1), device=Q.device)

    # Outer loop: Load a block of Keys and Values into "SRAM"
    for j in range(0, N, block_size_j):
        K_block = K[j : j + block_size_j]  # Shape: [block_size_j, D]
        V_block = V[j : j + block_size_j]  # Shape: [block_size_j, D]

        # Inner loop: Load a block of Queries into "SRAM" and match against KV
        for i in range(0, N, block_size_i):
            Q_block = Q[i : i + block_size_i]  # Shape: [block_size_i, D]

            # --- STEP A: Compute raw local attention logits ---
            S_local = torch.matmul(Q_block, K_block.T) * scale  # [block_size_i, block_size_j]

            # --- STEP B: Compute local row statistics ---
            m_local = torch.max(S_local, dim=1, keepdim=True)[0]  # [block_size_i, 1]
            P_local_unnormalized = torch.exp(S_local - m_local)   # [block_size_i, block_size_j]
            l_local = torch.sum(P_local_unnormalized, dim=1, keepdim=True)  # [block_size_i, 1]

            # --- STEP C: Load historical statistics for this query block ---
            m_old = m[i : i + block_size_i]  # [block_size_i, 1]
            l_old = l[i : i + block_size_i]  # [block_size_i, 1]
            O_old = O[i : i + block_size_i]  # [block_size_i, D]

            # --- STEP D: Compute new global statistics ---
            m_next = torch.max(m_old, m_local)  # [block_size_i, 1]

            # Compute numerical rescale factor deltas
            alpha = torch.exp(m_old - m_next)   # Scales old history
            beta = torch.exp(m_local - m_next)  # Scales incoming new block

            l_next = (alpha * l_old) + (beta * l_local)  # [block_size_i, 1]

            # --- STEP E: Rescale history and add current partial block product ---
            # We must isolate the raw matrix product multiplication before final division
            V_term = torch.matmul(P_local_unnormalized, V_block)  # [block_size_i, D]

            O_next = ((alpha * l_old * O_old) + (beta * V_term)) / l_next

            # --- STEP F: Write updated values back to Global Memory ---
            O[i : i + block_size_i] = O_next
            m[i : i + block_size_i] = m_next
            l[i : i + block_size_i] = l_next

    return O

# ==========================================================
# VERIFICATION PIPELINE
# ==========================================================
if __name__ == "__main__":
    # Generate random test inputs
    torch.manual_seed(42)
    seq_len, head_dim = 256, 64
    test_q = torch.randn(seq_len, head_dim)
    test_k = torch.randn(seq_len, head_dim)
    test_v = torch.randn(seq_len, head_dim)

    # 1. Compute via standard PyTorch textbook formulation (Exposes O(N^2) memory footprint)
    S_standard = torch.matmul(test_q, test_k.T) * (1.0 / (head_dim ** 0.5))
    A_standard = torch.softmax(S_standard, dim=-1)
    O_standard = torch.matmul(A_standard, test_v)

    # 2. Compute via our custom FlashAttention block implementation
    O_flash = flash_attention_forward(test_q, test_k, test_v, block_size_i=32, block_size_j=32)

    # 3. Assert mathematical equivalence
    is_correct = torch.allclose(O_standard, O_flash, atol=1e-5)
    print(f"Is FlashAttention mathematically identical to standard attention? -> **{is_correct}**")


Is FlashAttention mathematically identical to standard attention? -> **True**


Paged Attention

In [ ]:
rows = [
        {'address': '5412 N CLARK', 'date': '07/01/2012'},
        {'address': '5148 N CLARK', 'date': '07/04/2012'},
        {'address': '5800 E 58TH', 'date': '07/02/2012'},
        {'address': '2122 N CLARK', 'date': '07/03/2012'},
        {'address': '5645 N RAVENSWOOD', 'date': '07/02/2012'},
        {'address': '1060 W ADDISON', 'date': '07/02/2012'},
        {'address': '4801 N BROADWAY', 'date': '07/01/2012'},
        {'address': '1039 W GRANVILLE', 'date': '07/04/2012'},
]
from itertools import groupby
res = groupby(rows, key=lambda x: x['date'])
for date, items in res:
    print(date)
    for i in items:
        print('    ', i )

07/01/2012
     {'address': '5412 N CLARK', 'date': '07/01/2012'}
07/04/2012
     {'address': '5148 N CLARK', 'date': '07/04/2012'}
07/02/2012
     {'address': '5800 E 58TH', 'date': '07/02/2012'}
07/03/2012
     {'address': '2122 N CLARK', 'date': '07/03/2012'}
07/02/2012
     {'address': '5645 N RAVENSWOOD', 'date': '07/02/2012'}
     {'address': '1060 W ADDISON', 'date': '07/02/2012'}
07/01/2012
     {'address': '4801 N BROADWAY', 'date': '07/01/2012'}
07/04/2012
     {'address': '1039 W GRANVILLE', 'date': '07/04/2012'}


Per-Sample Gradients with vmap(grad)
Hard
jax
The capstone: compose two transforms. Standard backprop gives you the gradient of the mean loss — the per-example contributions are summed away. In JAX, per-example gradients are one composition: jax.vmap(jax.grad(...)).

For the linear model
$ŷ = w ̇x + b$
with per-sample squared error

ℓ
i
(
w
,
b
)
=
(
w
⋅
x
i
+
b
−
y
i
)
2
ℓ
i
​
 (w,b)=(w⋅x
i
​
 +b−y
i
​
 )
2


implement per_sample_grads(w, b, X, y) returning the pair (dW, db) where

dW has shape (N, D) — row
i
i is
∂
ℓ
i
/
∂
w
∂ℓ
i
​
 /∂w,
db has shape (N,) — entry
i
i is
∂
ℓ
i
/
∂
b
∂ℓ
i
​
 /∂b.
Write the loss for a single example, take jax.grad w.r.t. (w, b), then jax.vmap it over the rows of X and y (hint: in_axes=(None, None, 0, 0)). No Python loops.

Inputs: w — shape (D,); b — float; X — shape (N, D); y — shape (N,).

Example:
Input:
w = [1, 1], b = 0, X = [[1, 2], [3, 4]], y = [2, 6]
Output:
[[2.0, 4.0], [6.0, 8.0]]
[2.0, 2.0]
Reasoning:
Residuals: (1+2-2)=1 and (3+4-6)=1. Per-sample dW rows are 2·r_i·x_i = [2,4] and [6,8]; db entries are 2·r_i = [2, 2].



In [1]:
import jax
import jax.numpy as jnp


def _single_loss(w, b, x, y):
    """Per-example squared error: (w . x + b - y) ** 2"""
    pred = jnp.dot(w, x) + b
    return (pred - y) ** 2


def per_sample_grads(w, b, X, y):
    """
    Per-sample gradients of the squared-error loss, via vmap(grad(...)).

    Args:
        w: shape (D,)
        b: float
        X: shape (N, D)
        y: shape (N,)

    Returns:
        dW: shape (N, D) -- row i is d loss_i / d w
        db: shape (N,)   -- entry i is d loss_i / d b
    """
    grad_fn = jax.grad(_single_loss, argnums=(0, 1))
    batched_grad_fn = jax.vmap(grad_fn, in_axes=(None, None, 0, 0))
    dW, db = batched_grad_fn(w, b, X, y)
    return dW, db


In [2]:
  w = jnp.array([1.0, 1.0])
  b = 0.0
  X = jnp.array([[1.0, 2.0], [3.0, 4.0]])
  y = jnp.array([2.0, 6.0])

  dW, db = per_sample_grads(w, b, X, y)
  print(dW)
  print(db)

[[2. 4.]
 [6. 8.]]
[2. 2.]
